# FakeGuard — AI Fake News Detector
### NeuroLogic '26 Datathon

> Run cells one at a time. Each cell does one job only.
> If training fails after an epoch, do NOT restart from scratch — use the resume cell below.

### Beginner safety notes
- Use **GPU T4**, not P100.
- Run **Cell 1 and Cell 2 only once** at the very beginning.
- After a kernel restart, start from **Cell 3**.
- **Cell 8 is self-contained** and does not use `src/train.py`, so old cache issues do not matter.
- If training stops after epoch 1 or epoch 2, resume from the latest checkpoint instead of retraining.

### All known issues fixed
| Issue | Fix in notebook |
|---|---|
| `peft` import conflict | Cell 1 uninstalls peft |
| P100 incompatible with Kaggle PyTorch | Cell 3 blocks P100 and tells you to use T4 |
| `evaluation_strategy` deprecated | Cell 8 uses `eval_strategy` |
| `Trainer(tokenizer=...)` deprecated | Cell 8 removes it |
| Pickling errors from patching | No patching used anywhere |
| Old `src/train.py` cache problems | Cell 8 does not import `src/train.py` |

### Notebook flow
| Cell | Task | Time |
|---|---|---|
| 0 | Inspect competition dataset columns | instant |
| 1 | Fix packages | 30 sec |
| 2 | Restart kernel | instant |
| 3 | GPU check | instant |
| 4 | Clone/pull repo | 30 sec |
| 5 | Find dataset | instant |
| 6 | Preprocess data | 2 min |
| 7 | Baseline model | 2 min |
| 8 | Train RoBERTa | ~30 min |
| 8B | Resume training from checkpoint | only if needed |
| 9 | Evaluate model | 3 min |
| 10 | Generate predictions.csv | 2 min |
| 11 | Launch Gradio demo (live link for judges) | 1 min |


## Cell 0 — Inspect Competition Dataset Columns ⚠️ RUN THIS FIRST ON APRIL 25
Run this before anything else on competition day. It prints every CSV file it finds and shows you the exact column names.
This protects you if the competition dataset uses different column names (e.g. `headline` instead of `title`).

In [ ]:
import pandas as pd, glob
print('=== Scanning all CSV files in /kaggle/input ===')
matches = glob.glob('/kaggle/input/**/*.csv', recursive=True)
if not matches:
    print('No CSV files found. Please add your dataset via the + Add Data button on the right.')
else:
    for path in matches:
        try:
            df_peek = pd.read_csv(path, nrows=3)
            print(f'\n📂 File  : {path}')
            print(f'   Columns: {list(df_peek.columns)}')
            print(f'   Shape  : {df_peek.shape}')
            print(f'   Sample :')
            print(df_peek.head(2).to_string())
        except Exception as e:
            print(f'Could not read {path}: {e}')
print('\n=== Done. Check column names above before running Cell 6 ===')

## Cell 1 — Fix Package Conflict
Kaggle pre-installs `peft`, which conflicts with `transformers`. We do not use `peft`, so we remove it.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'peft'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers',
    'accelerate',
    'datasets',
    'seaborn',
])
print('Done. Now run Cell 2.')

## Cell 2 — Restart Kernel
Run once. After restart, skip Cells 1 and 2 and continue from Cell 3.

In [ ]:
import IPython
print('Restarting kernel... after restart begin from Cell 3.')
IPython.Application.instance().kernel.do_shutdown(restart=True)

## Cell 3 — GPU Check
This notebook must run on **Tesla T4**. P100 is not compatible with Kaggle's current PyTorch build.

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Go to Settings → Accelerator → GPU T4 x1')
gpu_name = torch.cuda.get_device_name(0)
print(f'GPU name: {gpu_name}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
if 'P100' in gpu_name:
    raise RuntimeError('P100 is not supported here. Switch to T4 in notebook settings.')
print('GPU is ready.')

## Cell 4 — Clone or Pull Repo

In [ ]:
import subprocess, os, sys
REPO_URL  = 'https://github.com/ayushtiwari18/neurologic-datathon-fakenews.git'
REPO_ROOT = '/kaggle/working/neurologic-datathon-fakenews'
if not os.path.exists(REPO_ROOT):
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_ROOT])
else:
    subprocess.check_call(['git', '-C', REPO_ROOT, 'pull'])
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
for d in ['data/raw', 'data/processed', 'outputs', 'models/roberta_fakenews']:
    os.makedirs(f'{REPO_ROOT}/{d}', exist_ok=True)
print(f'Repo ready: {os.getcwd()}')

## Cell 5 — Find Dataset

In [ ]:
import glob
from pathlib import Path
REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
MODEL_SAVE    = Path(f'{REPO_ROOT}/models/roberta_fakenews')
OUTPUTS_DIR   = Path(f'{REPO_ROOT}/outputs')
matches = glob.glob('/kaggle/input/**/*.csv', recursive=True)
if matches:
    DATASET_PATH = matches[0]
    print(f'Found dataset: {DATASET_PATH}')
else:
    DATASET_PATH = None
    raise FileNotFoundError('No CSV found in /kaggle/input. Use Add Data on the right panel.')

## Cell 6 — Preprocess Data

In [ ]:
import pandas as pd, shutil, sys, os
from pathlib import Path
REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
all_exist = all((PROCESSED_DIR / f).exists() for f in ['train.csv','val.csv','test.csv'])
if all_exist:
    train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
    val_df   = pd.read_csv(PROCESSED_DIR / 'val.csv')
    test_df  = pd.read_csv(PROCESSED_DIR / 'test.csv')
    print('Processed CSVs already exist. Loaded them.')
else:
    if 'DATASET_PATH' not in dir() or DATASET_PATH is None:
        raise ValueError('Run Cell 5 first.')
    shutil.copy2(DATASET_PATH, f'{REPO_ROOT}/data/raw/dataset.csv')
    from src.preprocess import run_preprocessing
    train_df, val_df, test_df = run_preprocessing()
print(f'Train : {len(train_df):,}')
print(f'Val   : {len(val_df):,}')
print(f'Test  : {len(test_df):,}')

## Cell 7 — Baseline Model

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
val_df   = pd.read_csv(PROCESSED_DIR / 'val.csv')
vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_df['combined'])
X_val   = vectorizer.transform(val_df['combined'])
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, train_df['label'])
baseline_preds = lr.predict(X_val)
baseline_acc   = accuracy_score(val_df['label'], baseline_preds)
print(f'Baseline Accuracy : {baseline_acc:.4f}')
print(classification_report(val_df['label'], baseline_preds, target_names=['FAKE','REAL']))

## Cell 8 — Train RoBERTa
This cell is self-contained and avoids all previous version issues.
If training finishes an epoch and later fails, use **Cell 8B** to resume from the last checkpoint instead of starting over.

In [ ]:
import os, json, torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
MODEL_SAVE    = Path(f'{REPO_ROOT}/models/roberta_fakenews')
OUTPUTS_DIR   = Path(f'{REPO_ROOT}/outputs')
MODEL_SAVE.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
class FakeNewsDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_length=512):
        self.df = pd.read_csv(csv_path)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(str(row['combined']), truncation=True, max_length=self.max_length, padding=False)
        return {
            'input_ids': enc['input_ids'],
            'attention_mask': enc['attention_mask'],
            'labels': int(row['label']),
        }
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1_score(labels, preds, average='weighted')}
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained('roberta-base')
print('Loading model...')
model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=2,
    id2label={0: 'FAKE', 1: 'REAL'},
    label2id={'FAKE': 0, 'REAL': 1},
)
train_dataset = FakeNewsDataset(PROCESSED_DIR / 'train.csv', tokenizer)
val_dataset   = FakeNewsDataset(PROCESSED_DIR / 'val.csv', tokenizer)
print(f'Train: {len(train_dataset):,} | Val: {len(val_dataset):,}')
training_args = TrainingArguments(
    output_dir=str(MODEL_SAVE),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to='none',
    logging_steps=100,
    seed=42,
    save_total_limit=2,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print('Training started...')
train_result = trainer.train()
eval_metrics = trainer.evaluate()
trainer.save_model(str(MODEL_SAVE))
tokenizer.save_pretrained(str(MODEL_SAVE))
metrics_out = {
    'final_val_accuracy': round(eval_metrics['eval_accuracy'], 6),
    'final_val_f1': round(eval_metrics['eval_f1'], 6),
    'train_runtime_seconds': round(train_result.metrics.get('train_runtime', 0), 2),
}
json.dump(metrics_out, open(OUTPUTS_DIR / 'training_metrics.json', 'w'), indent=2)
print(f"Val Accuracy : {eval_metrics['eval_accuracy']:.4f}")
print(f"Val F1       : {eval_metrics['eval_f1']:.4f}")
print('Training complete.')

## Cell 8B — Resume Training From Last Checkpoint (Only If Needed)
Use this **only if Cell 8 already completed at least one epoch** and then failed.
This saves time because it resumes from the last saved checkpoint instead of retraining from zero.

How it works:
- During training, Hugging Face saves folders like `checkpoint-1393`, `checkpoint-2786`, etc.
- This cell finds the newest checkpoint automatically.
- Then it resumes training from that checkpoint.

When to use it:
- Kernel disconnect after epoch 1 or epoch 2
- Notebook crash after at least one checkpoint was saved
- Temporary Kaggle failure

When NOT to use it:
- If Cell 8 failed before saving the first checkpoint
- If you changed the training code or model config and want a fresh run

In [ ]:
import os, json, torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
MODEL_SAVE    = Path(f'{REPO_ROOT}/models/roberta_fakenews')
OUTPUTS_DIR   = Path(f'{REPO_ROOT}/outputs')
checkpoints = sorted(MODEL_SAVE.glob('checkpoint-*'), key=os.path.getmtime)
if not checkpoints:
    raise FileNotFoundError('No checkpoint found. Use Cell 8 for a fresh run.')
latest_checkpoint = str(checkpoints[-1])
print(f'Resuming from: {latest_checkpoint}')
class FakeNewsDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_length=512):
        self.df = pd.read_csv(csv_path)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(str(row['combined']), truncation=True, max_length=self.max_length, padding=False)
        return {
            'input_ids': enc['input_ids'],
            'attention_mask': enc['attention_mask'],
            'labels': int(row['label']),
        }
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1_score(labels, preds, average='weighted')}
tokenizer = AutoTokenizer.from_pretrained('roberta-base')
model = RobertaForSequenceClassification.from_pretrained(
    latest_checkpoint,
    num_labels=2,
    id2label={0: 'FAKE', 1: 'REAL'},
    label2id={'FAKE': 0, 'REAL': 1},
)
train_dataset = FakeNewsDataset(PROCESSED_DIR / 'train.csv', tokenizer)
val_dataset   = FakeNewsDataset(PROCESSED_DIR / 'val.csv', tokenizer)
training_args = TrainingArguments(
    output_dir=str(MODEL_SAVE),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to='none',
    logging_steps=100,
    seed=42,
    save_total_limit=2,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
train_result = trainer.train(resume_from_checkpoint=latest_checkpoint)
eval_metrics = trainer.evaluate()
trainer.save_model(str(MODEL_SAVE))
tokenizer.save_pretrained(str(MODEL_SAVE))
metrics_out = {
    'final_val_accuracy': round(eval_metrics['eval_accuracy'], 6),
    'final_val_f1': round(eval_metrics['eval_f1'], 6),
    'train_runtime_seconds': round(train_result.metrics.get('train_runtime', 0), 2),
    'resumed_from': latest_checkpoint,
}
json.dump(metrics_out, open(OUTPUTS_DIR / 'training_metrics.json', 'w'), indent=2)
print(f"Val Accuracy : {eval_metrics['eval_accuracy']:.4f}")
print(f"Val F1       : {eval_metrics['eval_f1']:.4f}")
print('Resume training complete.')

## Cell 9 — Evaluate Model

In [ ]:
import sys, os
from pathlib import Path
from IPython.display import Image, display
REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.evaluate import run_evaluation
try:
    baseline_acc
except NameError:
    baseline_acc = 0.9466
eval_report = run_evaluation(
    val_csv_path=str(PROCESSED_DIR / 'val.csv'),
    baseline_accuracy=baseline_acc,
)
cm_path = f'{REPO_ROOT}/outputs/confusion_matrix.png'
if os.path.exists(cm_path):
    display(Image(cm_path))
print(f"RoBERTa Accuracy  : {eval_report['metrics']['accuracy']:.4f}")
print(f"Baseline Accuracy : {baseline_acc:.4f}")
print(f"Improvement       : +{eval_report['metrics']['accuracy'] - baseline_acc:.4f}")

## Cell 10 — Generate predictions.csv

In [ ]:
import sys, os
from pathlib import Path
REPO_ROOT     = '/kaggle/working/neurologic-datathon-fakenews'
PROCESSED_DIR = Path(f'{REPO_ROOT}/data/processed')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.predict import run_predict
predictions_df = run_predict(test_csv_path=str(PROCESSED_DIR / 'test.csv'))
print(f'Total : {len(predictions_df):,}')
print(f'FAKE  : {(predictions_df["predicted"] == 0).sum():,}')
print(f'REAL  : {(predictions_df["predicted"] == 1).sum():,}')
print('Saved to outputs/predictions.csv')

## Cell 11 — Launch Gradio Demo 🎯 (Live Link for Judges)
This launches the interactive FakeGuard demo with a **public shareable URL**.

- A link like `https://xxxx.gradio.live` will appear below after ~30 seconds
- Copy that link — paste it in your submission for judges to click
- The demo lets anyone type a news headline and instantly get REAL / FAKE / UNCERTAIN verdict
- **Keep this cell running** — if you stop it, the link goes offline

This scores Innovation (15%) and Real-World Impact (15%) judging points.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gradio>=4.0.0'])
print('Gradio installed.')

import sys, os
REPO_ROOT = '/kaggle/working/neurologic-datathon-fakenews'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from app.gradio_demo import launch_demo

print('='*60)
print('Launching FakeGuard demo...')
print('A public link will appear below in ~30 seconds.')
print('Copy the https://xxxx.gradio.live link for judges.')
print('='*60)

launch_demo(share=True)